## Inference on the QA Dataset (train.jsonl -> QA)

* google/bigbird-roberta-base w/o DA
* google/bigbird-roberta-base w DA on cleaned data
* google/bigbird-roberta-base w DA on conflicting data

In [1]:
from google.colab import drive

drive.mount('drive', force_remount=True)

Mounted at drive


In [2]:
%cd drive/MyDrive/Heidelberg/xtemp-nlp
!ls

/content/drive/MyDrive/Heidelberg/xtemp-nlp
conflict_planting	  model_hub_old  presentation	 run_mlm.ipynb
inference.ipynb		  new_data	 __pycache__	 run_mlm.py
inference-parallel.ipynb  old_data	 results	 run_mlm.sh
model_hub		  old_results	 run_eval_qa.py


In [3]:
import json
from collections import defaultdict
from string import Template
import numpy as np
import torch
from tqdm import tqdm
from collections import Counter
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForMaskedLM
import json
import os

In [4]:
def score_choice(choice, model, tokenizer):
    model.eval()
    device = model.device

    with torch.no_grad():
        q, a = choice.split(' <sep> ')
        q, a = q.strip(), a.strip()

        enc = tokenizer(q, a, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

        input_ids, attn_mask = enc.input_ids.to(device), enc.attention_mask.to(device)

        is_answer, answer_pos = False, []
        for idx, input_id in enumerate(input_ids[0]):
            # first [SEP]
            if not is_answer and input_id.item() == tokenizer.sep_token_id:
                is_answer = True
                continue
            # final [SEP]
            elif is_answer and input_id.item() == tokenizer.sep_token_id:
                break
            # answer is in-between [SEP] tokens
            if is_answer:
                answer_pos.append(idx)

        batch_input_ids, batch_attn_mask, target_token_ids = [], [], []
        for idx in answer_pos:
            token_id_original = input_ids[0, idx].item()

            masked = input_ids.clone()
            masked[0, idx] = tokenizer.mask_token_id

            batch_input_ids.append(masked[0])
            batch_attn_mask.append(attn_mask[0])
            target_token_ids.append(token_id_original)

        batch_input_ids = torch.stack(batch_input_ids).to(device)
        batch_attn_mask = torch.stack(batch_attn_mask).to(device)
        target_token_ids = torch.tensor(target_token_ids).to(device)

        outputs = model(input_ids=batch_input_ids, attention_mask=batch_attn_mask)

        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)

        token_logprobs = []

        for i, idx in enumerate(answer_pos):
            token_logprob = log_probs[i, idx, target_token_ids[i]].item()
            token_logprobs.append(token_logprob)

        logprob = sum(token_logprobs) / len(token_logprobs)

        return logprob, np.exp(logprob)

In [5]:
# model_id = 'google/bigbird-roberta-base'

eval_path = 'model_hub/bigbird-roberta-base-conflicts'

def get_model_tokenizer(model_id):


    model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True, device_map='auto')
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    return model, tokenizer


for checkpoint in os.listdir(eval_path):


    checkpoint_path = os.path.join(eval_path, checkpoint)

    ## evaluate only last checkpoint due to upcoming deadline
    if not os.path.isdir(checkpoint_path):
       continue
    if checkpoint != 'checkpoint-1672':
       continue

    print(f'Processing checkpoint {checkpoint}')

    m, t = get_model_tokenizer(model_id=checkpoint_path)


    labels = [1, 2, 3, 4]
    prompt_template = Template('$question <sep> $answer')
    Y, Y_hat, per_domain_metrics = [], [], defaultdict(list)

    with open('new_data/test.json', 'r') as f:
        ds = json.load(f)

        print(f"\n\n*** Evaluate QA ***\nNum Questions : {len(ds)}\n\n")
        corr, limit, total = 0.0, 0, 0
        for entry in tqdm(ds, desc='Inference on full QA dataset + Paraphrases'):
            q = entry['question']
            q_paraphrased = entry['q-paraphrased']

            choices = [prompt_template.substitute(question=q, answer=entry[f'op{clet}']) for clet in ['a', 'b', 'c', 'd']]
            choices_paraphrased = [prompt_template.substitute(question=q_paraphrased, answer=entry[f'op{clet}']) for clet in ['a', 'b', 'c', 'd']]

            cop = entry['cop']

            max_prob, ans = float('-inf'), None
            max_prob_paraphrase, ans_paraphrase = float('-inf'), None

            for cop_idx in range(len(choices)):
                logs, _ = score_choice(choices[cop_idx], m, t)
                if max_prob < logs:
                    max_prob = logs
                    ans = cop_idx + 1

            for cop_idx in range(len(choices_paraphrased)):
                logs, _ = score_choice(choices_paraphrased[cop_idx], m, t)
                if max_prob_paraphrase < logs:
                    max_prob_paraphrase = logs
                    ans_paraphrase = cop_idx + 1

            if ans == ans_paraphrase:
              Y_hat.append(ans)
            else:
              ## if the predictions don't agree, we choose incorrect answer
              for opt in labels:
                if opt != cop:
                    Y_hat.append(opt)
                    ans = opt
                    break

            Y.append(cop)


            if ans == cop:
              per_domain_metrics[entry['subject_name']].append('1')
              corr += 1
            else:
              per_domain_metrics[entry['subject_name']].append('0')

            total += 1
            limit += 1

            ## the accuracy is printed every 500 examples
            if limit == 500:
               print(f'overall accuracy: {round(corr / total, 4)}')
               limit = 0


    p_macro = precision_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)
    r_macro = recall_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)
    f1_macro = f1_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)

    acc = accuracy_score(Y, Y_hat)

    report = classification_report(Y, Y_hat, zero_division=0, labels=labels,
                                  target_names=["opa", "opb", "opc", "opd"])

    report_dict = classification_report(Y, Y_hat, zero_division=0, labels=labels,
                                  target_names=["opa", "opb", "opc", "opd"], output_dict=True)

    print(f'Gold Ans: {Counter(Y)}')
    print(f'Pred Ans: {Counter(Y_hat)}')

    print(f'\n\nReport\n\n{report_dict}')

    d = {'P-macro': p_macro, 'R-macro': r_macro,
        'F1-macro': f1_macro, 'Acc': acc, 'Report': report_dict}

    print(d)

    print('\n\n--- PER DOMAIN METRICS ---\n\n')

    for domain, instances in per_domain_metrics.items():
        print(f'{domain} Instances: {len(instances)} Acc: {round(instances.count('1') / len(instances), 4)}\n')
        per_domain_metrics[domain] = {'Acc':round(instances.count('1') / len(instances), 4), 'Support': len(instances)}

    with open(f'results/conflicts/{checkpoint}.json', 'w') as f:
        json.dump(d, f, indent=2)

    with open(f'results/conflicts/{checkpoint}-per-domain-metrics.json', 'w') as f:
        json.dump(per_domain_metrics, f, indent=2)



Processing checkpoint checkpoint-1672


BigBirdForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BigBirdForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.




*** Evaluate QA ***
Num Questions : 14540




...

overall accuracy: 0.2348


Inference on full QA dataset + Paraphrases: 100%|█████████▉| 14501/14540 [37:07<00:06,  6.38it/s]

overall accuracy: 0.2343


Inference on full QA dataset + Paraphrases: 100%|██████████| 14540/14540 [37:13<00:00,  6.51it/s]


Gold Ans: Counter({1: 4400, 2: 3695, 3: 3310, 4: 3135})
Pred Ans: Counter({1: 4815, 2: 3677, 4: 3100, 3: 2948})


Report

{'opa': {'precision': 0.2157840083073728, 'recall': 0.23613636363636364, 'f1-score': 0.22550189907759088, 'support': 4400.0}, 'opb': {'precision': 0.22110416100081587, 'recall': 0.22002706359945873, 'f1-score': 0.22056429734129138, 'support': 3695.0}, 'opc': {'precision': 0.25407055630936226, 'recall': 0.22628398791540785, 'f1-score': 0.23937360178970918, 'support': 3310.0}, 'opd': {'precision': 0.25903225806451613, 'recall': 0.256140350877193, 'f1-score': 0.2575781876503609, 'support': 3135.0}, 'accuracy': 0.2341127922971114, 'macro avg': {'precision': 0.23749774592051676, 'recall': 0.2346469415071058, 'f1-score': 0.23575449646473806, 'support': 14540.0}, 'weighted avg': {'precision': 0.23517669751490386, 'recall': 0.2341127922971114, 'f1-score': 0.23432102302787414, 'support': 14540.0}}
{'P-macro': 0.23749774592051676, 'R-macro': 0.2346469415071058, 'F1-macro': 0.